# 异常处理 + 文件 IO —— 进阶（习题）

Easy

1. 自定义一个异常类 NegativeValueError(继承 Exception)。写函数 check_positive(n):n 为负就 raise NegativeValueError,否则返回 n。测试正数、负数、0。

In [10]:
class NegativeValueError(Exception):
    pass

def check_positive(n):
    if n < 0:
        raise NegativeValueError(f"{n}为负数")
    else:
        return n

try:
    print(check_positive(8))
    print(check_positive(0))
    print(check_positive(-1))
except NegativeValueError as e:
    print(f"输入错误：{e}")


8
0
输入错误：-1为负数


参考答案

自定义异常 + raise + 捕获都对。一个小细节:if n < 0: raise 后面的 else: 可以省掉——raise 会中断函数,走到 return n 就说明没 raise。else 不算错,但 Day 9 我提过「卫语句」风格:

In [35]:
def check_positive(n):
    if n < 0:
        raise NegativeValueError(f"{n}为负数")
    return n          # 不用 else,raise 已经中断了

2. 写函数 parse_with_else(s):用 try/except/else 结构把字符串转 int。成功在 else 里打印「成功」并返回值,失败返回 None。体会 else 和把逻辑塞在 try 里的区别。

In [16]:
def parse_with_else(s):
    try:
        value = int(s)
    except ValueError:
        print("转化失败")
        return None
    else:
        print("成功")
        return value

In [17]:
parse_with_else("1")

成功


1

In [15]:
parse_with_else("d")

转化失败


3. 用 pathlib 写函数 file_info(path_str):返回一个 dict,包含文件是否存在、文件名、后缀。对 ../data/sales.csv 和一个不存在的路径分别测试。

In [19]:
from pathlib import Path

def file_info(path_str):
    path = Path(path_str)
    return {"文件存在性": path.exists(),"文件名":path.name,"后缀":path.suffix}

file_info("../data/sales.csv")

{'文件存在性': True, '文件名': 'sales.csv', '后缀': '.csv'}

In [20]:
file_info("../datasets")

{'文件存在性': False, '文件名': 'datasets', '后缀': ''}

Medium

4. —(重做 Day 9 题 5,这次做对) 写 read_numbers(path_str):读每行一个数字的文件,返回数字列表。要求用 pathlib 检查存在性、文件不存在返回 []、坏行跳过并 logging.warning 记录。这次必须测「文件不存在」这个用例。

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

def read_numbers(path_str):
    path = Path(path_str)
    lst = []
    
    if not path.exists():
        logging.warning(f"文件不存在：{path}")
        return []
    
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try: 
                lst.append(float(line))
            except ValueError as e:
                logging.warning(f"跳过坏数据 {line}: {e}")
                continue
        return lst
        
read_numbers("nums_file.txt")

2026-05-28 14:46:07,829 [WARNING] 跳过坏数据 abc
: could not convert string to float: 'abc\n'


[123.0, 234.0, 345.0]

参考答案

代码逻辑正确,文件不存在返回 []、坏行跳过 + log 都做到了。这次把 Day 9 漏掉的 FileNotFoundError 场景补上了,进步明显。 但:
问题 1(小 bug):return lst 缩进错了,在 with 块内。

In [ ]:
# with path.open("r", encoding="utf-8") as f:
#         for line in f:
#             ...
#     return lst        # ← 这个 return 在 with 内、for 外

这次碰巧没出错,因为 for 跑完正好到这。但语义上 return 应该在 with 块外面(读完、文件关闭后再返回)。虽然结果一样,但 return 放 with 内会让人误以为「还没读完就返回」。挪出去:

In [38]:
# with path.open("r", encoding="utf-8") as f:
#         for line in f:
#             try:
#                 lst.append(float(line))
#             except ValueError as e:
#                 logging.warning(f"跳过坏数据 {line.strip()}: {e}")
#     return lst            # ← with 外,读完且文件已关闭

问题 2(细节):log 里的 line 带换行符。 看你的输出:

In [39]:
# [WARNING] 跳过坏数据 abc
# : could not convert string to float: 'abc\n'

那个换行让日志断成两行,很丑。line 是 "abc\n",带着文件里的换行符。logging 里用 line.strip() 去掉首尾空白:f"跳过坏数据 {line.strip()}: {e}"。顺便,float() 其实能处理带换行的字符串(float("123\n") = 123.0),所以你的解析没问题,只是日志显示难看。

补一句边界用例(你这题没列):你测了「坏行」和「不存在的文件」,但没测空文件(应返回 [])和全是坏行的文件(应返回 [] + 几条 warning)。下次记得把这两个也造出来。

5. 写 write_report(data, path_str):把一个 dict 列表用 csv.DictWriter 写成 CSV 文件(记得 newline="")。然后读回来验证。fieldnames 从第一条记录的 keys 自动取。

In [25]:
import csv

def write_report(data, path_str):
    path = Path(path_str)

    if not path.exists():
        logging.warning(f"文件不存在：{path}")
    
    with open(data, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=data[0].keys())
        writer.writeheader()
        writer.writerows(data)

    with path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for line in reader:
            print(line)

参考答案

Bug 1(致命):open(data, ...) 把 data(那个 dict 列表)当成文件路径传给了 open。 data 是 [{...}, {...}],不是路径字符串。这行会直接抛 TypeError。应该是 open(path, ...) 或 path.open(...)。
你这题 cell 没有输出——说明你根本没运行它(只定义了函数没调用)。如果调用了,立刻就会炸。这又回到弱点 #19:写完一定要跑、要测。定义了不调用 = 没验证 = 不知道对不对。

Bug 2(逻辑):写文件前检查 path.exists() 是反的。 你是要写一个新文件,写之前文件「不存在」是正常情况,不该 warning。这个检查应该删掉——写文件不需要它。(检查存在性是「读」文件时才做的事。)
正确版:

In [40]:
def write_report(data, path_str):
    path = Path(path_str)
    if not data:                          # 该检查的是:data 是不是空的
        logging.warning("data 为空,不写文件")
        return
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=data[0].keys())
        writer.writeheader()
        writer.writerows(data)
    # 读回验证
    with path.open("r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            print(row)

# 必须实际调用测试!
write_report([{"name": "Alice", "score": 90}, {"name": "Bob", "score": 85}], "scores.csv")

{'name': 'Alice', 'score': '90'}
{'name': 'Bob', 'score': '85'}


记住:fieldnames=data[0].keys() 有个隐含前提——data 不能是空列表,否则 data[0] 会 IndexError。所以开头先判 if not data。这就是边界思维。

6. 写 parse_price(s):用异常链——内部 float(s) 失败时,raise ValueError(...) from e 包装。调用方捕获后能打印根本原因 e.__cause__。

In [ ]:
def parse_price(s):
    try:
        return float(s)
    except ValueError as e:
        raise ValueError(f"格式错误：{s}") from e
        

try:
    parse_price("a")
except ValueError as e:
    print(f"根本错误：{e.__cause__}")

根本错误：could not convert string to float: 'a'


7. —(改造 Day 9 题 10) 重写 process_sales(records):按 country 分组求和。要求用 logging.warning 记录每条被跳过的坏数据(而不是 print,也不是默默跳过)。自己造测试数据,故意混入缺 country、total 非数字的坏记录。

In [30]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

def process_sales(records):
    total_country_sales = {}

    for record in records:
        try:
            country = record["country"]
            total = record["total"]
            total_country_sales[country] = total_country_sales.get(country, 0) + total
        except (KeyError, ValueError, TypeError) as e:
            logging.warning(f"跳过坏数据 {record}: {e}")
            continue
    return total_country_sales

In [31]:
test_orders = [
    # 正常记录1
    {"country": "中国", "total": 100, "order_id": "001"},
    # 正常记录2
    {"country": "美国", "total": 200, "order_id": "002"},
    # 坏数据1：缺 country 字段
    {"total": 150, "order_id": "003"},
    # 坏数据2：total 是字符串 "abc"，无法转为数字
    {"country": "日本", "total": "abc", "order_id": "004"},
    # 正常记录3：同国家，验证累加逻辑
    {"country": "中国", "total": 50, "order_id": "005"},
    # 正常记录4：浮点数销售额，验证数值转换
    {"country": "德国", "total": 89.9, "order_id": "006"},
]

In [32]:
process_sales(test_orders)

2026-05-28 15:14:15,270 [WARNING] 跳过坏数据 {'total': 150, 'order_id': '003'}: 'country'
2026-05-28 15:14:15,271 [WARNING] 跳过坏数据 {'country': '日本', 'total': 'abc', 'order_id': '004'}: unsupported operand type(s) for +: 'int' and 'str'


{'中国': 150, '美国': 200, '德国': 89.9}

Hard

8. 写 csv_to_json(csv_path, json_path):读 CSV(用 csv.DictReader),把 quantity/price/total 三列从字符串转成数字(转不了的行 logging.warning 跳过),整理后写成 JSON。要全程健壮:文件不存在、坏行都不能让程序崩。用 data/sales.csv 实测。

In [33]:
import json

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

def csv_to_json(csv_path, json_path):
    path = Path(csv_path)
    
    if not path.exists():
        logging.warning(f"文件不存在：{path}")
    
    results = []

    with path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

        for row in rows:
            try:
                row["quantity"] = int(row["quantity"])
                row["price"] = float(row["price"])
                row["total"] = float(row["total"])

                results.append(row)
            except ValueError as e: 
                logging.warning(f"跳过坏数据 {row}: {e}")
                continue

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

In [34]:
csv_to_json("../data/sales.csv", "sales.json")

参考答案

主体对:DictReader 读、三列转数字、坏行 log 跳过、写 JSON。csv_to_json("../data/sales.csv", "sales.json") 实际跑了(没报错没输出是正常的,因为函数没 print)。

改进 1:文件不存在时,你只 warning 了没 return。

In [42]:
# if not path.exists():
#     logging.warning(f"文件不存在：{path}")
    # ← 没有 return!继续往下走会在 path.open() 炸

文件不存在你 warning 完,代码继续往下到 with path.open(...),这里会抛 FileNotFoundError 崩掉。题目要求「文件不存在不能崩」。补 return:

In [43]:
# if not path.exists():
#     logging.warning(f"文件不存在：{path}")
#     return        # ← 直接返回,别往下走

隐患 2:int(row["quantity"]) 缺键时抛 KeyError,你只抓了 ValueError。 如果某行缺 quantity 列,row["quantity"] 抛 KeyError,但你 except ValueError 抓不到 → 崩。改成 except (ValueError, KeyError) 更稳。sales.csv 列齐全所以没触发,又是「碰巧对」。

这题你没列边界测试用例(题目硬性要求)。至少该测:① 正常文件 ② 不存在的文件 ③ 自己造一个含坏行的小 CSV。下次补。

9. 写一个自己的上下文管理器 Timer:用 class + __enter__/__exit__,这样 with Timer(): ... 能自动打印代码块耗时。(提示:__enter__ 记开始时间,__exit__ 算耗时。)这是 Day 9 装饰器 timer 的「with 版」——对照着理解。

之前没学过class，有点超纲太多了

10. —(综合大题 + 边界清单) 写 sales_report(csv_path):读 sales.csv,生成一份按 country 的销售报告 dict(每个国家:订单数、总额、平均额),写成 JSON 文件。要求:

• 全程异常安全(文件不存在、坏行、空文件都不崩),坏数据 logging.warning

• 空文件时平均额不能除零崩溃

• (b) 在 cell 末尾列出你测了哪些边界用例(至少 4 种),并说明每种的预期行为

太难了，有点太复杂了。